# Are the pure groups just singletons?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
from pathlib import Path

import plotly.graph_objects as go
import polars as pl
from IPython.display import Markdown, display

from fraud_detection.evaluation.entity_purity import (
    Anchor,
    EntityKey,
    compare,
    purity,
)


def get_raw_dir():
    import os
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "kaggle" / "raw").exists():
            return current / "kaggle" / "raw"
        current = current.parent
    return Path("../../kaggle/raw")

COLS = ["TransactionID", "TransactionDT", "isFraud", "card1", "addr1", "D1"]
raw_dir = get_raw_dir()
df = pl.scan_csv(raw_dir / "train_transaction.csv", infer_schema_length=10000, null_values=[""]).select(COLS).collect()
display(Markdown(f"**Dataset loaded:** {len(df):,} rows"))


**Dataset loaded:** 590,540 rows

In [2]:
card = EntityKey(columns=("card1",), name="card1")
card_addr = EntityKey(columns=("card1", "addr1"), name="card1+addr1")
client = EntityKey(columns=("card1", "addr1"), anchors=(Anchor("D1"),), name="client uid")

comp_df = compare(df, [card, card_addr, client], label="isFraud")[
    ["entity", "entities", "entities_multi", "pure_share_multi", "pure_share_all",
     "singleton_share", "fraud_share_in_touched_groups"]
]

display(Markdown(comp_df.to_pandas().to_markdown(index=False)))


| entity      |   entities |   entities_multi |   pure_share_multi |   pure_share_all |   singleton_share |   fraud_share_in_touched_groups |
|:------------|-----------:|-----------------:|-------------------:|-----------------:|------------------:|--------------------------------:|
| card1       |      13553 |            10109 |             0.848  |           0.8866 |            0.2541 |                          0.249  |
| card1+addr1 |      37531 |            22713 |             0.8958 |           0.937  |            0.3948 |                          0.363  |
| client uid  |     199070 |            83557 |             0.9853 |           0.9938 |            0.5803 |                          0.8489 |

In [3]:
labels = comp_df['entity'].to_list()
pure_share = comp_df['pure_share_multi'].to_list()
fraud_share = comp_df['fraud_share_in_touched_groups'].to_list()

fig = go.Figure(data=[
    go.Bar(name='Purity (pure_share_multi)', x=labels, y=pure_share, marker_color='#66b3ff', text=[f"{v*100:.1f}%" for v in pure_share], textposition='auto'),
    go.Bar(name='Fraud Strength (fraud_share_in_touched_groups)', x=labels, y=fraud_share, marker_color='#ff9999', text=[f"{v*100:.1f}%" for v in fraud_share], textposition='auto')
])

fig.update_layout(barmode='group', title='Client Key Evolution (Higher is better)',
                  yaxis_title='Percentage', template='plotly_white')
fig.update_yaxes(range=[0, 1.15])
fig.show()


#### Conclusion
 
The `client_uid` significantly improves the `fraud_share_in_touched_groups` metric (from ~25% for `card1` to ~85%), proving that it effectively isolates true clients. In groups flagged by a single fraud transaction, the vast majority of other transactions are also fraud.



#### The Singleton Problem

A singleton group (1 transaction) is 100% pure by definition. Metrics like `pure_share_all` are artificially inflated by singletons and should be ignored. The primary metric must be `pure_share_multi` (purity for groups with >1 transaction).

In [4]:
absurd = EntityKey(columns=("TransactionID",), name="one entity per transaction")
p = purity(df, absurd.assign(df), "isFraud")

table = (
    f"| Metric | Value | Meaning |\n"
    f"| --- | --- | --- |\n"
    f"| `pure_share_all` | **{p.pure_share_all}** | Perfect score, heavily inflated by singletons |\n"
    f"| `pure_share_multi` | **{p.pure_share_multi}** | No entity has two transactions to compare |\n"
)
display(Markdown(table))


| Metric | Value | Meaning |
| --- | --- | --- |
| `pure_share_all` | **1.0** | Perfect score, heavily inflated by singletons |
| `pure_share_multi` | **nan** | No entity has two transactions to compare |
